In [26]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'gcd'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    if not os.path.exists(result_file):
        continue
    df = pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['fold_type'] = df['args'].apply(lambda x: x['fold_type'])
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    if method == 'plm_ood':
        df['method'] = df.apply(lambda x: 'Llama3.1-8B' + '-' + x['args']['Detecor'], axis=1)
        # print()
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx', 'fold_type'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx', 'fold_type'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_6": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_type', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 783/783 [00:00<00:00, 98209.93it/s]


100%|██████████| 784/784 [00:00<00:00, 131538.63it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,fold_type,metric,value
21030,0.1,0.25,20NG,ALUP,0,fold,ACC,94.35
21034,0.1,0.25,20NG,ALUP,0,fold,ARI,90.45
21031,0.1,0.25,20NG,ALUP,0,fold,H-Score,96.14
21032,0.1,0.25,20NG,ALUP,0,fold,K-ACC,100.00
21033,0.1,0.25,20NG,ALUP,0,fold,N-ACC,92.56
...,...,...,...,...,...,...,...,...
19918,1.0,0.75,thucnews,TAN,4,fold,ARI,77.16
19915,1.0,0.75,thucnews,TAN,4,fold,H-Score,78.68
19916,1.0,0.75,thucnews,TAN,4,fold,K-ACC,95.96
19917,1.0,0.75,thucnews,TAN,4,fold,N-ACC,66.67


In [27]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_type', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                         0.1                                \
known_cls_ratio                      0.25                                 
metric                                ACC    ARI H-Score   K-ACC  N-ACC   
dataset  method fold_type fold_idx                                        
20NG     ALUP   fold      0         94.35  90.45   96.14  100.00  92.56   
                          1         94.28  95.19   96.04  100.00  92.38   
                          2         99.80  99.54   99.77   99.71  99.83   
                          3         94.75  91.24   96.12   99.21  93.22   
                          4         93.61  89.34   95.35   99.75  91.32   
...                                   ...    ...     ...     ...    ...   
thucnews TAN    fold      0         69.54  54.88   72.97   92.89  60.08   
                          1         76.40  66.53   80.05   96.78  68.25   
                          2         72.58  60.41   76.31   94.74  63.88   
                          3         64.17  53.01   66.09   71.93  61.12   
                          4         67.56  50.79   70.55   82.67  61.52   

labeled_ratio                                                              \
known_cls_ratio                              0.50                           
metric                                NMI     ACC     ARI H-Score   K-ACC   
dataset  method fold_type fold_idx                                          
20NG     ALUP   fold      0         93.46   98.99   98.05   99.02   99.57   
                          1         98.26   99.87   99.69   99.87   99.86   
                          2         99.63  100.00  100.00  100.00  100.00   
                          3         94.52   99.53   99.06   99.52   99.87   
                          4         92.92   99.73   99.49   99.72  100.00   
...                                   ...     ...     ...     ...     ...   
thucnews TAN    fold      0         67.96   75.69   64.98   71.11   94.34   
                          1         76.74   79.65   68.40   77.37   93.08   
                          2         72.05   70.39   55.63   64.48   91.93   
                          3         66.96   79.58   64.09   78.19   90.10   
                          4         64.99   83.25   68.49   82.56   90.58   

labeled_ratio                       ...     1.0                                \
known_cls_ratio                     ...    0.50                          0.75   
metric                              ... H-Score   K-ACC  N-ACC    NMI     ACC   
dataset  method fold_type fold_idx  ...                                         
20NG     ALUP   fold      0         ...   99.87  100.00  99.74  99.76   99.93   
                          1         ...   99.94  100.00  99.87  99.88  100.00   
                          2         ...   99.93  100.00  99.87  99.88  100.00   
                          3         ...   99.60   99.61  99.59  99.40   99.87   
                          4         ...   99.93  100.00  99.86  99.88   96.30   
...                                 ...     ...     ...    ...    ...     ...   
thucnews TAN    fold      0         ...   64.63   97.03  48.45  72.11   77.17   
                          1         ...   74.56   96.33  60.82  75.75   86.86   
                          2         ...   76.72   94.67  64.49  73.82   88.06   
                          3         ...   72.03   96.04  57.63  73.60   77.60   
                          4         ...   76.13   94.09  63.92  75.18   87.70   

labeled_ratio                                                               
known_cls_ratio                                                             
metric                                 ARI H-Score   K-ACC   N-ACC     NMI  
dataset  method fold_type fold_idx                                          
20NG     ALUP   fold      0          99.85   99.88  100.00   99.76   99.88  
                          1         100.00  100.00  100.00  100.00  100.00  
                          2         100.

In [28]:
# 基于 df_melted = [labeled_ratio, known_cls_ratio, dataset, method, fold_idx, metric, value]

# 认为：只要某个 metric 有记录，就视为该 (dataset, method, labeled_ratio, known_cls_ratio, fold_idx) 已测试
cells = (df_melted[["dataset","method","labeled_ratio","known_cls_ratio","fold_type", "fold_idx"]]
         .drop_duplicates()
         .sort_values(["dataset","method","labeled_ratio","known_cls_ratio","fold_type", "fold_idx"])
         .reset_index(drop=True))

# 1) 每个 dataset × method 的维度覆盖与计数
progress_by_dm = (cells
    .groupby(["dataset","method", "fold_type"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),  # 完成的组合总数
    )
    .reset_index()
)

# 2) 每个 dataset × method × labeled_ratio × known_cls_ratio 下完成了多少个 fold
fold_coverage = (cells
    .groupby(["dataset","method","labeled_ratio","known_cls_ratio","fold_type"])["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset","method","labeled_ratio","known_cls_ratio"])
)

In [29]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method","fold_type"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio","fold_type"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["fold_type", "dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio                                    0.1                   \
known_cls_ratio                                 0.25             0.50   
fold_type      dataset  method                                          
fold           20NG     ALUP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DPN          [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DeepAligned  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        GeoID        [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        LOOP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                              ...              ...   
imbalance_fold thucnews DeepAligned           [0, 1]              NaN   
                        GeoID                 [0, 1]              NaN   
                        LOOP                  [0, 1]              NaN   
                        PLM_GCD               [0, 1]              NaN   
                        SDC                   [0, 1]              NaN   

labeled_ratio                                                     0.5  \
known_cls_ratio                                 0.75             0.25   
fold_type      dataset  method                                          
fold           20NG     ALUP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DPN          [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DeepAligned  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        GeoID        [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        LOOP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                              ...              ...   
imbalance_fold thucnews DeepAligned              NaN              NaN   
                        GeoID                    NaN              NaN   
                        LOOP                     NaN              NaN   
                        PLM_GCD                  NaN              NaN   
                        SDC                      NaN              NaN   

labeled_ratio                                                          \
known_cls_ratio                                 0.50             0.75   
fold_type      dataset  method                                          
fold           20NG     ALUP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DPN          [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DeepAligned  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        GeoID        [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        LOOP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                              ...              ...   
imbalance_fold thucnews DeepAligned              NaN              NaN   
                        GeoID                    NaN              NaN   
                        LOOP                     NaN              NaN   
                        PLM_GCD                  NaN              NaN   
                        SDC                      NaN              NaN   

labeled_ratio                                    1.0                   \
known_cls_ratio                                 0.25             0.50   
fold_type      dataset  method                                          
fold           20NG     ALUP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DPN          [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        DeepAligned  [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        GeoID        [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
                        LOOP         [0, 1, 2, 3, 4]  [0, 1, 2, 3, 4]   
...                                              ...              ...   
imbalance_fold thucnews DeepAligned              NaN              NaN   
                        GeoID                    NaN              NaN   
                        LOOP                     NaN              NaN   
                        PLM_GCD                  NaN   

In [30]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.9134920634920635

In [31]:
all_num

5040

In [32]:
cur_num

4604.0

In [33]:
# # 原始处理
# tmp_df = df_melted.groupby(
#     ['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'metric']
# ).mean().reset_index()

# tmp_df = tmp_df[
#     (tmp_df['labeled_ratio'] == 0.1)
#     & (tmp_df['dataset'].isin(['banking', 'clinc', 'hwu', 'mcid', 'stackoverflow']))
#     & (tmp_df['metric'].isin(['ACC', 'NMI', 'K-ACC', 'N-ACC']))
# ]

# tmp_df = tmp_df[
#     tmp_df['method'].isin(['DeepAligned', 'DPN', 'GeoID', 'TAN', 'SDC', 'LOOP', 'ALUP', 'PLM_GCD'])
# ]

# # ===== 设置排序顺序 =====
# dataset_order = ['banking', 'clinc', 'hwu', 'mcid', 'stackoverflow']
# metric_order = ['ACC', 'NMI', 'K-ACC', 'N-ACC']
# method_order = ['DeepAligned', 'DPN', 'GeoID', 'TAN', 'SDC', 'LOOP', 'ALUP', 'PLM_GCD']

# # 转换为分类变量（有序）
# tmp_df['dataset'] = pd.Categorical(tmp_df['dataset'], categories=dataset_order, ordered=True)
# tmp_df['metric'] = pd.Categorical(tmp_df['metric'], categories=metric_order, ordered=True)
# tmp_df['method'] = pd.Categorical(tmp_df['method'], categories=method_order, ordered=True)

# # ===== 生成透视表并按顺序输出 =====
# tmp_df_pivot = tmp_df.pivot(
#     index=['known_cls_ratio', 'method'],
#     columns=['dataset', 'metric'],
#     values='value'
# )

# # 重新排序 index & columns
# tmp_df_pivot = tmp_df_pivot.sort_index(
#     axis=0, level=['known_cls_ratio', 'method']
# ).sort_index(
#     axis=1, level=['dataset', 'metric']
# )

# tmp_df_pivot

In [34]:
# ===== 设置排序顺序 =====
dataset_order = ['banking', 'clinc', 'hwu', 'mcid', 'stackoverflow']
metric_order = ['K-F1', 'N-F1']
method_order = ['DOC', 'DeepUnk', 'ADB', 'clap', 'KnnCon', 'DyEn', 'Llama3.1-8B-EnergyBased', 'Llama3.1-8B-Entropy',
       'Llama3.1-8B-KLMatching', 'Llama3.1-8B-LogitNorm',
       'Llama3.1-8B-Mahalanobis', 'Llama3.1-8B-MaxLogit',
       'Llama3.1-8B-MaxSoftmax', 'Llama3.1-8B-OpenMax',
       'Llama3.1-8B-TemperatureScaling']

# 原始处理
tmp_df = df_melted.groupby(
    ['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'metric']
).mean().reset_index()

tmp_df = tmp_df[
    (tmp_df['labeled_ratio'] == 1.0)
    & (tmp_df['dataset'].isin(dataset_order))
    & (tmp_df['metric'].isin(metric_order))
]

tmp_df = tmp_df[
    tmp_df['method'].isin(method_order)
]


# 转换为分类变量（有序）
tmp_df['dataset'] = pd.Categorical(tmp_df['dataset'], categories=dataset_order, ordered=True)
tmp_df['metric'] = pd.Categorical(tmp_df['metric'], categories=metric_order, ordered=True)
tmp_df['method'] = pd.Categorical(tmp_df['method'], categories=method_order, ordered=True)

# ===== 生成透视表并按顺序输出 =====
tmp_df_pivot = tmp_df.pivot(
    index=['known_cls_ratio', 'method'],
    columns=['dataset', 'metric'],
    values='value'
)

# 重新排序 index & columns
tmp_df_pivot = tmp_df_pivot.sort_index(
    axis=0, level=['known_cls_ratio', 'method']
).sort_index(
    axis=1, level=['dataset', 'metric']
)

tmp_df_pivot

TypeError: agg function failed [how->mean,dtype->object]